In [2]:
import scanpy as sc
import pickle
import os
import anndata
import mudata
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import ArrowStyle
from scipy.stats import fisher_exact
import seaborn as sns
import glob
import PyComplexHeatmap as pch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib_venn import venn2
from ast import literal_eval
from scipy.stats import hypergeom
from scipy.stats import pearsonr
import numpy as np
from scipy.stats import pearsonr, spearmanr
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import SpectralCoclustering,SpectralBiclustering
from sklearn.metrics import consensus_score
from statsmodels.stats.multitest import multipletests
import glob as glob


In [20]:
outdir="/data2st2/junyi/output/sn0615"

# Read the meta data 

In [5]:
if os.path.exists("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv"):
    df_meta = pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv", index_col=0)
    df_meta.drop_duplicates(inplace=True)
else:
    df_meta = pd.DataFrame()
    h5ads = glob.glob('/data2st2/junyi/output/stg1028/*_4VN/*.h5ad')
    for h5ad in h5ads:
        adata = sc.read_h5ad(h5ad, backed='r')
        df_meta = pd.concat([df_meta, pd.DataFrame(adata.obs)], axis=0)
    df_meta.to_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv")
    df_meta.drop_duplicates(inplace=True)

/tmp/ipykernel_3091102/3001891177.py:2: DtypeWarning: Columns (20,56,57,58,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta = pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv", index_col=0)


In [58]:
df_meta

,sample,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,...,umap_seurat_1,umap_seurat_2,celltype,Neurotransmitter,region_celltypeL2_condition,condition,Region Subclass,bacode,ctename,ctname
AAACCCAAGCTTGTTG-1PFC_beirui_FC63E,FC63E_PFC_beirui,4453.0,8.401558,16167.0,9.690789,30.259170,37.551803,46.180491,59.473001,7.0,...,-8.439108,-5.629186,PFC L5 IT Glut,Glut,NaN,NaN,PFC_PFC L5 IT Glut,AAACCCAAGCTTGTTG-1,PFC_PFC L5 IT Glut,PFC_PFC_L5_IT_Glut
AAACCCAAGTATGATG-1PFC_beirui_FC63E,FC63E_PFC_beirui,3150.0,8.055475,8260.0,9.019301,28.704600,35.786925,44.443099,58.716707,16.0,...,-8.133489,-0.168780,PFC L4/5 IT Glut,Glut,NaN,NaN,PFC_PFC L4/5 IT Glut,AAACCCAAGTATGATG-1,PFC_PFC L4/5 IT Glut,PFC_PFC_L4_5_IT_Glut
AAACCCACAACCCTAA-1PFC_beirui_FC63E,FC63E_PFC_beirui,2836.0,7.950502,7621.0,8.938794,33.105892,39.837292,47.841491,61.592967,5.0,...,-3.175797,8.722422,PFC L2/3 IT Glut,Glut,NaN,NaN,PFC_PFC L2/3 IT Glut,AAACCCACAACCCTAA-1,PFC_PFC L2/3 IT Glut,PFC_PFC_L2_3_IT_Glut
AAACCCACACATACGT-1PFC_beirui_FC63E,FC63E_PFC_beirui,5370.0,8.588769,21106.0,9.957360,30.299441,37.060552,44.868758,57.130674,12.0,...,-3.936474,7.770447,PFC L2/3 IT Glut,Glut,NaN,NaN,PFC_PFC L2/3 IT Glut,AAACCCACACATACGT-1,PFC_PFC L2/3 IT Glut,PFC_PFC_L2_3_IT_Glut
AAACCCACATCGATGT-1PFC_beirui_FC63E,FC63E_PFC_beirui,1715.0,7.447751,3077.0,8.032036,25.901852,32.694183,42.508937,60.513487,6.0,...,-1.848519,-2.865366,MOL-3,NN,NaN,NaN,PFC_MOL-3,AAACCCACATCGATGT-1,PFC_MOL-3,PFC_MOL-3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTTCTCGGGAC-1STR_beirui_CS1-1,CS1-1_STR_beirui,1973.0,7.587817,3907.0,8.270781,25.620681,33.401587,42.820578,60.122856,3.0,...,5.922817,-11.399503,STR D1 Chst9 GABA,GABA,NaN,NaN,STR_STR D1 Chst9 GABA,TTTGGTTTCTCGGGAC-1,STR_STR D1 Chst9 GABA,STR_STR_D1_Chst9_GABA
TTTGTTGAGTCCTGTA-1STR_beirui_CS1-1,CS1-1_STR_beirui,1044.0,6.951772,1736.0,7.459915,32.776498,39.976959,51.382488,68.663594,1.0,...,-1.794904,-6.987206,Astrocyte-2,NN,NaN,NaN,STR_Astrocyte-2,TTTGTTGAGTCCTGTA-1,STR_Astrocyte-2,STR_Astrocyte-2
TTTGTTGAGTTCCGGC-1STR_beirui_CS1-1,CS1-1_STR_beirui,1712.0,7.446001,3052.0,8.023880,26.179554,33.125819,42.267366,60.288336,0.0,...,6.191682,-11.427277,STR D1 Chst9 GABA,GABA,NaN,NaN,STR_STR D1 Chst9 GABA,TTTGTTGAGTTCCGGC-1,STR_STR D1 Chst9 GABA,STR_STR_D1_Chst9_GABA
TTTGTTGCATAAGCGG-1STR_beirui_CS1-1,CS1-1_STR_beirui,3036.0,8.018625,7566.0,8.931552,30.980703,37.734602,45.717684,59.344436,0.0,...,-2.728244,-14.239271,STR Maf_Egfr GABA,GABA,NaN,NaN,STR_STR Maf_Egfr GABA,TTTGTTGCATAAGCGG-1,STR_STR Maf_Egfr GABA,STR_STR_Maf_Egfr_GABA


In [26]:
# Read the meta data and create a map between desired maps
df_meta['Region Subclass'] = df_meta['region'] + "_" + df_meta['celltype.L2']
df_meta['ctname'] = df_meta['Region Subclass'].str.replace(" ","_")
df_meta['ctname'] = df_meta['ctname'].str.replace("/","_")
df_meta['bacode'] = df_meta.index.str[:18]

In [13]:
df_meta.status.value_counts()

status
SUS      557716
CON      540042
CSRES    291287
RES      285130
CSDS     214690
Name: count, dtype: int64

In [33]:
df_meta_M3R = df_meta[df_meta['region'].isin(['AMY', 'HPF', 'PFC'])].copy()
df_meta_M3R = df_meta_M3R[df_meta_M3R['status'].isin(['SUS', 'CON'])]
df_meta_M3R = df_meta_M3R[df_meta_M3R.sex == 'M']

In [48]:
df_selected_sample = df_meta_M3R.drop_duplicates(subset=['sample'])

In [50]:
df_selected_sample.to_csv(os.path.join(outdir, "selected_samples.csv"), index=False)

In [36]:
# 为每个 sample 创建文件夹，并为每个 ctname 生成 barcode 表
import os

# 使用 df_meta_M3R (AMY/HIP/PFC, MALE, SUS/CON)
# 为每个 sample 创建文件夹
sample_list = df_meta_M3R['sample'].unique()
print(f"样本数: {len(sample_list)}")
print(f"样本列表: {sample_list}")

# 在 outdir 下创建 sample 文件夹
for sample in sample_list:
    sample_dir = os.path.join(outdir, "sample_barcodes", sample)
    os.makedirs(sample_dir, exist_ok=True)
    
    # 获取该 sample 的所有 cell
    sample_df = df_meta_M3R[df_meta_M3R['sample'] == sample]
    
    # 为该 sample 中的每个 ctname 生成一个 barcode 表
    for ctname, ct_df in sample_df.groupby('ctname'):
        # 提取 barcode，每一行一个 barcode
        barcodes = ct_df[['bacode']].drop_duplicates()
        outfile = os.path.join(sample_dir, f"{ctname}_barcodes.csv")
        barcodes.to_csv(outfile, header=False, index=False)
    
    print(f"  {sample}: {sample_df['ctname'].nunique()} 个 celltype")

print("\n完成！每个 sample 文件夹下包含各 ctname 的 barcode 表。")


样本数: 24
样本列表: ['MW26B_PFC_novogene' 'MW26E_PFC_yunzhun' 'MW45A_AMY_yunzhun'
 'MW45C_AMY_beirui' 'MW45C_HPF_beirui' 'MW26E_AMY_yunzhun'
 'MW51A_HPF_yunzhun' 'MW22B_AMY_yunzhun' 'MW45A_HPF_yunzhun'
 'MW22B_HPF_yunzhun' 'MW34C_PFC_beirui' 'MW51A_PFC_yunzhun'
 'MC33A_HPF_yunzhun' 'MC37A_AMY_yunzhun' 'MC25B_PFC_yunzhun'
 'MC48E_HPF_beirui' 'MC21D_PFC_beirui' 'MC33A_PFC_novogene'
 'MC48E_AMY_beirui' 'MC48D_HPF_yunzhun' 'MC52E_PFC_yunzhun'
 'MC25B_HPF_yunzhun' 'MC25B_AMY_yunzhun' 'MC48D_AMY_yunzhun']
  MW26B_PFC_novogene: 38 个 celltype
  MW26E_PFC_yunzhun: 35 个 celltype
  MW45A_AMY_yunzhun: 40 个 celltype
  MW45C_AMY_beirui: 39 个 celltype
  MW45C_HPF_beirui: 36 个 celltype
  MW26E_AMY_yunzhun: 40 个 celltype
  MW51A_HPF_yunzhun: 35 个 celltype
  MW22B_AMY_yunzhun: 37 个 celltype
  MW45A_HPF_yunzhun: 41 个 celltype
  MW22B_HPF_yunzhun: 35 个 celltype
  MW34C_PFC_beirui: 34 个 celltype
  MW51A_PFC_yunzhun: 34 个 celltype
  MC33A_HPF_yunzhun: 40 个 celltype
  MC37A_AMY_yunzhun: 44 个 celltype
  MC25B_PFC_y

In [51]:
bamfiles = glob.glob("/data1st2/mark/snRNA/*/cellranger_output*/*/cellranger_output/*/outs/possorted_genome_bam.bam")

In [54]:
# 从 bamfiles 中找出属于 df_selected_sample 的样本
# 注意: HPF 在路径中对应的是 HIP
region_map = {"HPF": "HIP"}  # 其他 region 名保持一致

# 建立 bam 文件路径的 sample -> path 映射
bam_dict = {}
for b in bamfiles:
    parts = b.split("/")
    sample_name = parts[8]  # 路径中第8个元素是 sample 名
    bam_dict[sample_name] = b

print(f"bam 文件中共有 {len(bam_dict)} 个唯一样本")

# 遍历 df_selected_sample 中的每个样本，匹配 bam 文件
selected_bam = []
for _, row in df_selected_sample.iterrows():
    sample = row['sample']
    found = False
    
    # 直接匹配
    if sample in bam_dict:
        selected_bam.append({"sample": sample, "bam_path": bam_dict[sample], "match_type": "exact"})
        found = True
    else:
        # 尝试替换 region: HPF -> HIP
        for old, new in region_map.items():
            alt_sample = sample.replace(old, new)
            if alt_sample in bam_dict:
                selected_bam.append({"sample": sample, "bam_path": bam_dict[alt_sample], "match_type": f"{old}->{new}"})
                found = True
                break
    
    if not found:
        print(f"  ❌ {sample} 未找到对应的 bam 文件")
        selected_bam.append({"sample": sample, "bam_path": None, "match_type": "missing"})

df_selected_bam = pd.DataFrame(selected_bam)
print(f"\n成功匹配 {df_selected_bam['bam_path'].notna().sum()} / {len(df_selected_bam)} 个样本的 bam 文件")

# 按匹配类型统计
print("\n匹配方式:")
for t, count in df_selected_bam['match_type'].value_counts().items():
    print(f"  {t}: {count}")

# 展示结果
print("\n匹配结果:")
for _, r in df_selected_bam.iterrows():
    status = "✅" if r['bam_path'] else "❌"
    print(f"  {status} {r['sample']} → {r['bam_path'] if r['bam_path'] else '未找到'}")


bam 文件中共有 163 个唯一样本

成功匹配 24 / 24 个样本的 bam 文件

匹配方式:
  exact: 16
  HPF->HIP: 8

匹配结果:
  ✅ MW26B_PFC_novogene → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_novogene_out/cellranger_output/MW26B_PFC_novogene/outs/possorted_genome_bam.bam
  ✅ MW26E_PFC_yunzhun → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_yunzhun_out/cellranger_output/MW26E_PFC_yunzhun/outs/possorted_genome_bam.bam
  ✅ MW45A_AMY_yunzhun → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_yunzhun_out/cellranger_output/MW45A_AMY_yunzhun/outs/possorted_genome_bam.bam
  ✅ MW45C_AMY_beirui → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/MW45C_AMY_beirui/outs/possorted_genome_bam.bam
  ✅ MW45C_HPF_beirui → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw/snRNA_batch_beirui_out/cellranger_output/MW45C_HIP_beirui/outs/possorted_genome_bam.bam
  ✅ MW26E_AMY_yunzhun → /data1st2/mark/snRNA/snRNA_CUMS/cellranger_output_raw

In [55]:
df_selected_bam

,sample,bam_path,match_type
0,MW26B_PFC_novogene,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
1,MW26E_PFC_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
2,MW45A_AMY_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
3,MW45C_AMY_beirui,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
4,MW45C_HPF_beirui,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,HPF->HIP
5,MW26E_AMY_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
6,MW51A_HPF_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,HPF->HIP
7,MW22B_AMY_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,exact
8,MW45A_HPF_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,HPF->HIP
9,MW22B_HPF_yunzhun,/data1st2/mark/snRNA/snRNA_CUMS/cellranger_out...,HPF->HIP


In [57]:
df_selected_sample.merge(df_selected_bam, on='sample', how='left').to_csv(os.path.join(outdir, "selected_samples_with_bam.csv"), index=False)